In [ ]:
""" Created on August 22, 2025 // Updated on March 20, 2026 // @author: Sarah Shi """

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mineralML as mm

%matplotlib inline
%config InlineBackend.figure_format = 'png'

# mineralML Stoichiometry Calculations

This notebook demonstrates how to use the **stoichiometry** calculators and classifiers defined in mineralML:
1. Load and prepare data for analysis
2. Calculate stoichiometry with `BaseMineralCalculator`, for moles, oxygens, cations on a fixed oxygen basis
3. Apply specialized calculators for different mineral groups
4. Perform empirical classifications consistent with petrologist-defined schemes

We loaded in the ``mineralML`` Python package as ``mm``. ``mineralML`` has trained machine learning models for classifying minerals. This implementation aims to get your electron microprobe or quantitative EDS compositions classified and processed. We remove some degrees of freedom to simplify the process as much as possible. The minerals considered for this study include: Amphibole, Apatite, Biotite, Calcite, Chlorite, Epidote, Feldspar (Alkali Feldspar and Plagioclase), Garnet, Glass, Kalsilite, Leucite, Melilite, Muscovite, Nepheline, Olivine, Oxide (Rhombohedral_Oxides including Hematite-Ilmenite, Spinel_Group including Magnetite-Spinel), Pyroxene (Clinopyroxene, Orthopyroxene, Na-Pyroxene), Quartz, Rutile, Serpentine, Titanite, Tourmaline, and Zircon. 

One CSV file containing your electron microprobe analyses in oxide weight percentages is necessary. Find an example [here](https://github.com/sarahshi/mineralML/blob/main/docs/examples/training_hundred.csv). The necessary oxides are SiO$_2$, TiO$_2$, Al$_2$O$_3$, FeO$_t$, MnO, MgO, CaO, Na$_2$O, K$_2$O, Cr$_2$O$_3$, P$_2$O$_5$, and ZrO$_2$ (if you are aiming to classify zircon). For the oxides not analyzed for specific minerals, the preprocessing will fill in the nan values as 0. 

# BaseMineralCalculator

``mm.BaseMineralCalculator`` forms the foundation for all stoichiometry workflows in ``mineralML``. It provides the core methods to:

- Normalize oxide compositions to a fixed oxygen basis
- Convert weight% oxides to moles, oxygens, and cations
- Enforce Fe input consistency (FeOt, Fe₂O₃t, or paired FeO/Fe₂O₃)

It returns results in a consistent format: 
- \_mols
- \_ox
- \_cat_{oxbasis}ox, cations per n oxygens

All mineral-specific calculators (e.g., AmphiboleCalculator, FeldsparCalculator, OlivineCalculator, PyroxeneCalculator) inherit from this base. They extend it with additional logic for site allocation, normalization rules, or classification schemes. For most users, no direct interaction is needed with BaseMineralCalculator,  but it is useful to know that every group-specific tool builds on this common calculator.

## 1. Load and prepare data for analysis

In [ ]:
# Read in your dataframe of mineral data, called training_hundred.csv. 
df_load = mm.load_df('TabularData/training_hundred.csv')
display(df_load.head())

## AmphiboleCalculator and AmphiboleClassifier

``mm.AmphiboleCalculator`` and ``mm.AmphiboleClassifier`` perform basic calculations associated with amphibole-group minerals along with site allocations (:cite:t:`Leakeetal1997` and :cite:t:`Ridolfi2021` from ``Thermobar`` :cite:p:`Wieseretal2022`), and additionally plots these data in a classification diagram. ``mineralML`` adds on to the ``Thermobar`` classification by also classifies amphiboles on the :cite:t:`Leakeetal1997` calcic amphibole classification diagram and explicitly returns the corresponding ``Submineral``, or the specific type of calcic amphibole. 

In [ ]:
# Let's select just the amphiboles from the training dataset
amp = df_load[df_load.Mineral=='Amphibole']
amp_calc = mm.AmphiboleCalculator(amp)
amp_comp = amp_calc.calculate_components()

In [ ]:
# Let's print all the columns in this dataframe, along with the compositions returned.
print(list(amp_comp.columns))
display(amp_comp.head())

In [ ]:
# Now, let's classify and plot up these amphiboles. 
amp_class = mm.AmphiboleClassifier(amp)
amp_comp_class = amp_class.classify()

# Note the addition of the Submineral column providing a calcic amphibole classification.
display(amp_comp_class.head())

In [ ]:
# Here, we can plot up these amphiboles. 
fig, ax = amp_class.plot()

## ApatiteCalculator

``mm.ApatiteCalculator`` performs basic calculations and site allocations associated with apatite-group minerals.

In [ ]:
ap = df_load[df_load.Mineral=='Apatite']
ap_calc = mm.ApatiteCalculator(ap)
ap_comp = ap_calc.calculate_components()
display(ap_comp.head())

## BiotiteCalculator

``mm.BiotiteCalculator`` performs basic calculations and site allocations associated with biotite-group minerals.

In [ ]:
bt = df_load[df_load.Mineral=='Biotite']
bt_calc = mm.BiotiteCalculator(bt)
bt_comp = bt_calc.calculate_components()
display(bt_comp.head())

## CalciteCalculator

``mm.CalciteCalculator`` performs basic calculations and site allocations associated with calcite minerals.

In [ ]:
cal = df_load[df_load.Mineral=='Carbonate']
cal_calc = mm.CalciteCalculator(cal)
cal_comp = cal_calc.calculate_components()
display(cal_comp.head())

## ChloriteCalculator

``mm.ChloriteCalculator`` performs basic calculations and site allocations associated with chlorite minerals.

In [ ]:
chl = df_load[df_load.Mineral=='Chlorite']
chl_calc = mm.ChloriteCalculator(chl)
chl_comp = chl_calc.calculate_components()
display(chl_comp.head())

## ClinopyroxeneCalculator, OrthopyroxeneCalculator, PyroxeneClassifier

``mm.ClinopyroxeneCalculator``, ``mm.OrthopyroxeneCalculator``, and ``mm.PyroxeneClassifier`` perform basic calculations associated with pyroxene-group minerals along with site allocations and additionally plots these data in a ternary diagram. ``mineralML`` adds on to the ``Thermobar`` ternary diagram by also classifies pyroxenes on the :cite:t:`DHZ` pyroxene ternary diagram and explicitly returns the corresponding ``Submineral``, or the specific type of pyroxene. 

The ``mm.PyroxeneClassifier`` is recommended if you do not know what types of pyroxenes are present. The specific calculators can be used if the type of pyroxene is known.

In [ ]:
# Let's select just the pyroxenes from the training dataset
px = df_load[(df_load.Mineral=='Clinopyroxene') | (df_load.Mineral=='Orthopyroxene')]
px_class = mm.PyroxeneClassifier(px)
px_comp = px_class.calculate_components()
display(px_comp.head())

In [ ]:
# Plot these pyroxenes! 
fig, ax = px_class.plot()

## EpidoteCalculator

``mm.EpidoteCalculator`` performs basic calculations and site allocations associated with epidote minerals.

In [ ]:
# Let's select just the epidotes from the training dataset
ep = df_load[df_load.Mineral=='Epidote']
ep_calc = mm.EpidoteCalculator(ep)
ep_comp = ep_calc.calculate_components()
display(ep_comp.head())

## FeldsparCalculator, FeldsparClassifier

``mm.FeldsparCalculator`` and ``mm.FeldsparClassifier`` perform basic calculations associated with feldspar-group minerals along with site allocations and additionally plots these data in a ternary diagram. ``mineralML`` pulls the lines from the ``Thermobar`` ternary diagram, and further classifies data on the feldspar ternary diagram and explicitly returns the corresponding ``Submineral``, or the specific type of feldspar. 

In [ ]:
# Let's select just the feldspars from the training dataset
feld = df_load[(df_load.Mineral=='KFeldspar') | (df_load.Mineral=='Plagioclase')]
feld_class = mm.FeldsparClassifier(feld)
feld_comp = feld_class.calculate_components()
display(feld_comp.head())

In [ ]:
# Plot these feldspars! 
fig, ax = feld_class.plot()

## GarnetCalculator

``mm.GarnetCalculator`` performs basic calculations and site allocations associated with garnet minerals. This includes the Droop calculation for determining the proportion of Fe³⁺ and Fe²⁺. 

In [ ]:
# Let's select just the garnets from the training dataset
gt = df_load[df_load.Mineral=='Garnet']
gt_calc = mm.GarnetCalculator(gt)
gt_comp = gt_calc.calculate_components()
display(gt_comp.head())

## KalsiliteCalculator

``mm.KalsiliteCalculator`` performs basic calculations and site allocations associated with kalsilite minerals. 

In [ ]:
# Let's select just the kalsilite from the training dataset
kal = df_load[df_load.Mineral=='Kalsilite']
kal_calc = mm.KalsiliteCalculator(gt)
kal_comp = kal_calc.calculate_components()
display(kal_comp.head())

## LeuciteCalculator

``mm.LeuciteCalculator`` performs basic calculations and site allocations associated with leucite minerals. 

In [ ]:
# Let's select just the leucite from the training dataset
lc = df_load[df_load.Mineral=='Leucite']
lc_calc = mm.LeuciteCalculator(gt)
lc_comp = lc_calc.calculate_components()
display(lc_comp.head())

## MeliliteCalculator

``mm.MeliliteCalculator`` performs basic calculations and site allocations associated with melilite minerals. 

In [ ]:
# Let's select just the melilite from the training dataset
ml = df_load[df_load.Mineral=='Melilite']
ml_calc = mm.MeliliteCalculator(ml)
ml_comp = ml_calc.calculate_components()
display(ml_comp.head())

## MuscoviteCalculator

``mm.MuscoviteCalculator`` performs basic calculations and site allocations associated with muscovite minerals. 

In [ ]:
# Let's select just the muscovite from the training dataset
ms = df_load[df_load.Mineral=='Muscovite']
ms_calc = mm.MuscoviteCalculator(ms)
ms_comp = ms_calc.calculate_components()
display(ms_comp.head())

## NephelineCalculator

``mm.NephelineCalculator`` performs basic calculations and site allocations associated with nepheline minerals. 

In [ ]:
# Let's select just the nepheline from the training dataset
ne = df_load[df_load.Mineral=='Nepheline']
ne_calc = mm.NephelineCalculator(ne)
ne_comp = ne_calc.calculate_components()
display(ne_comp.head())

## OlivineCalculator

``mm.OlivineCalculator`` performs basic calculations and site allocations associated with olivine minerals. 

In [ ]:
# Let's select just the olivine from the training dataset
ol = df_load[df_load.Mineral=='Olivine']
ol_calc = mm.OlivineCalculator(ol)
ol_comp = ol_calc.calculate_components()
display(ol_comp.head())

## RhombohedralOxideCalculator, SpinelCalculator, OxideClassifier

``mm.RhombohedralOxideCalculator``, ``mm.SpinelCalculator``, and ``mm.OxideClassifier`` perform basic calculations associated with oxide and spinel minerals along with site allocations  and additionally plots these data in a ternary diagram. This includes the Droop calculation for determining the proportion of Fe³⁺ and Fe²⁺. ``mineralML`` classifies oxides on the :cite:t:`DHZ` Ti⁴⁺-R³⁺-R²⁺ ternary diagram and explicitly returns the corresponding ``Submineral``, or the specific type of oxides, and ``Subspinel``, if the mineral is a spinel.

The ``mm.OxideClassifier`` is recommended if you do not know what types of oxides are present. The specific calculators can be used if the type of oxide is known. If there are spinels detected, the data will also be plotted on a spinel classification diagram.

In [ ]:
# Let's select just the oxides from the training dataset
ox = df_load[(df_load.Mineral=='Hematite') | (df_load.Mineral=='Ilmenite') | (df_load.Mineral=='Spinel') | (df_load.Mineral=='Magnetite')]
ox_class = mm.OxideClassifier(ox)
ox_comp = ox_class.calculate_components()
display(ox_comp.head())

In [ ]:
# Plot these oxides! 
fig, ax = ox_class.plot()

## QuartzCalculator

``mm.QuartzCalculator`` performs basic calculations and site allocations associated with quartz minerals. 

In [ ]:
# Let's select just the quartz from the training dataset
qz = df_load[df_load.Mineral=='SiO2_Polymorph']
qz_calc = mm.QuartzCalculator(qz)
qz_comp = qz_calc.calculate_components()
display(qz_comp.head())

## RutileCalculator

``mm.RutileCalculator`` performs basic calculations and site allocations associated with rutile minerals. 

In [ ]:
# Let's select just the rutile from the training dataset
rt = df_load[df_load.Mineral=='Rutile']
rt_calc = mm.RutileCalculator(rt)
rt_comp = rt_calc.calculate_components()
display(rt_comp.head())

## SerpentineCalculator

``mm.SerpentineCalculator`` performs basic calculations and site allocations associated with serpentine minerals. 

In [ ]:
# Let's select just the serpentine from the training dataset
srp = df_load[df_load.Mineral=='Serpentine']
srp_calc = mm.SerpentineCalculator(srp)
srp_comp = srp_calc.calculate_components()
display(srp_comp.head())

## TitaniteCalculator

``mm.TitaniteCalculator`` performs basic calculations and site allocations associated with titanite minerals. 

In [ ]:
# Let's select just the titanite from the training dataset
tit = df_load[df_load.Mineral=='Titanite']
tit_calc = mm.TitaniteCalculator(tit)
tit_comp = tit_calc.calculate_components()
display(tit_comp.head())

## TourmalineCalculator

``mm.TourmalineCalculator`` performs basic calculations and site allocations associated with tourmaline minerals. 

In [ ]:
# Let's select just the tourmaline from the training dataset
trm = df_load[df_load.Mineral=='Tourmaline']
trm_calc = mm.TourmalineCalculator(trm)
trm_comp = trm_calc.calculate_components()
display(trm_comp.head())

## ZirconCalculator

``mm.ZirconCalculator`` performs basic calculations and site allocations associated with zircon minerals. 

In [ ]:
# Let's select just the zircon from the training dataset
zr = df_load[df_load.Mineral=='Zircon']
zr_calc = mm.ZirconCalculator(zr)
zr_comp = zr_calc.calculate_components()
display(zr_comp.head())